# 01 Von der Frage zur strukturierten Antwort

**Reihe:** LangChain durch Literate Programming lernen, Notebook 1 von 4.

**Leitfrage:** Wie kann ein Modul-Assistent eine Frage anhand eines kurzen Textes beantworten
und dabei eine Quelle angeben?

Dieses Notebook beginnt mit einer Frage zur mitgelieferten Bitcoin-Unterlage.
Schritt für Schritt entsteht daraus eine Verarbeitungskette: Die Anfrage wird vorbereitet,
an das Modell übergeben und die Antwort in eine feste Form gebracht.

Am Ende enthält das Ergebnis zwei Felder: **Antwort** und **Quelle**.
Der Text wird zunächst aus der ersten Seite dieser PDF gelesen. Ein Gesprächsverlauf folgt in Notebook 02,
die automatische Suche in PDF-Unterlagen in Notebook 03.

LangChain stellt die Bausteine bereit, mit denen sich diese Schritte verbinden lassen.
Die folgenden Abschnitte führen sie einzeln ein.

### Orientierung für den Einstieg

## Lernziele

Nach dem Durcharbeiten können Leserinnen und Leser:

- die Rollen von System-, Human- und AI-Nachrichten unterscheiden und erklären, warum eine neue Anfrage den bisherigen Verlauf nicht automatisch kennt;
- einen Prompt mit Platzhaltern erstellen und Prompt, Modell und Parser zu einer Chain verbinden;
- eine vollständige Antwort mit einer schrittweise übertragenen Streaming-Antwort vergleichen;
- gemeldete Tokenzahlen lesen und erklären, warum ohne Tarif kein Geldbetrag berechnet werden kann.

## Konzept Erklärung

Das Modell erhält eine Frage und einen Text. Dieser Text heisst **Kontext**. Eine Regel sagt zum Beispiel: „Antworte nur anhand dieses Textes.“ Regel, Kontext und Frage bilden den Prompt.

LangChain verpackt den Prompt in **Nachrichten**. Jede Nachricht hat eine Rolle:

| Rolle | Bedeutung hier |
|---|---|
| System | Regeln für die Antwort. |
| Human | Text und Frage der nutzenden Person. |
| AI | Zurückgegebene Modellantwort. |

**Text und Frage → Nachrichten → BFH-Modell → Antwortnachricht**

Eine neue Anfrage enthält frühere Antworten nicht automatisch. Wenn sie gebraucht werden, müssen sie erneut mitgeschickt werden. Notebook 02 ergänzt diesen Verlauf.

### Der Weg zur Antwort als Bild

![Regeln, Text und Frage werden durch die Vorlage zu Nachrichten. Das BFH-Modell erzeugt eine Antwort. Der Parser liest diese ein und prüft die Felder antwort und quelle.](abbildungen/01_von_der_frage_zur_antwort.svg)

Das Bild zeigt den gewünschten Ablauf der Kette am Ende dieses Notebooks. Die **Vorlage** setzt die Eingaben zusammen. Das **Modell** formuliert die Antwort mit den verlangten Feldern. Der **Parser** liest und prüft diese Felder; er entscheidet nicht, ob die Aussage fachlich richtig ist. Zuerst werden diese Schritte einzeln angesehen.


## Setup

Die Zellen werden von oben nach unten ausgeführt. Dabei bleiben Variablen im laufenden
Python-Kernel verfügbar. Jede folgende Codezelle hat zunächst nur eine kleine Aufgabe.

Zuerst werden Python-Hilfsmittel geladen: `os` liest Einstellungen, `sys` liefert die
Python-Version und `Path` bildet Dateipfade. `version` liest Paketversionen aus.
`load_dotenv` lädt später die lokale Datei `.env`. Es wird noch kein Modell aufgerufen.

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

Die Versionsprüfung verhindert einen Start mit einer nicht unterstützten Python-Version.
Die beiden Ausgaben helfen später bei der Suche nach Installationsproblemen.

In [2]:
assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 oder 3.13 erforderlich'
print('Python:', sys.version.split()[0])
print('LangChain Core:', version('langchain-core'))

Python: 3.13.15
LangChain Core: 1.7.3


Die Projektdateien werden über relative Pfade gefunden. Jupyter kann im Projektordner
oder direkt im Ordner `notebooks` gestartet sein. Im zweiten Fall führt `.parent`
eine Ebene nach oben. Ein persönlicher Benutzerpfad ist dadurch nicht nötig.

In [3]:
projektordner = Path.cwd()
if projektordner.name == 'notebooks':
    projektordner = projektordner.parent
assert (projektordner / 'modul_assistent' / 'core.py').is_file(), 'Im Projekt oder in notebooks starten'
if str(projektordner) not in sys.path:
    sys.path.insert(0, str(projektordner))


Die Datei `.env` enthält lokale BFH-Einstellungen. `load_dotenv` übernimmt sie in die Umgebung des Python-Prozesses. `override=False` bedeutet: Bereits gesetzte Werte bleiben erhalten.

Eingabe ist der Dateipfad. Ergebnis sind verfügbare Einstellungen. Es wird noch keine Anfrage gesendet. Der Schlüssel wird nicht angezeigt.


In [4]:
load_dotenv(projektordner / '.env', override=False)


True

## Experiment

### 1 Die Bitcoin Unterlage finden

Die erste Frage bezieht sich auf `Eigenschaften-Bitcoin.pdf`. Diese einseitige
Modulunterlage liegt im Repository unter `data/modulunterlagen`. Die zweite PDF
behandelt Open Banking und kommt später bei der gemeinsamen Dokumentensuche hinzu.

Der Pfad beginnt beim Projektordner. Die Prüfung mit `is_file()` zeigt frühzeitig,
ob die Datei am erwarteten Ort liegt. Ausgegeben wird nur ihr Dateiname.

In [5]:
pdf_datei = projektordner / 'data' / 'modulunterlagen' / 'Eigenschaften-Bitcoin.pdf'
assert pdf_datei.is_file(), 'Die Bitcoin-Unterlage fehlt in data/modulunterlagen.'
print('Unterlage:', pdf_datei.name)

Unterlage: Eigenschaften-Bitcoin.pdf


### 2 Den Text der ersten Seite auslesen

`PdfReader` aus dem Paket `pypdf` öffnet die PDF. `pages[0]` bedeutet die erste Seite,
weil Python bei null zu zählen beginnt. `extract_text()` liest ihren Text aus.
Dieser Text wird in `kontext` gespeichert und zur Kontrolle angezeigt.

Hier wird noch keine passende Textstelle gesucht. Die ganze erste Seite wird als
bekannte Informationsbasis verwendet. Das spätere RAG-Notebook ergänzt die Suche.

In [6]:
from pypdf import PdfReader

pdf = PdfReader(pdf_datei)
kontext = pdf.pages[0].extract_text()
assert kontext and kontext.strip(), 'Aus der PDF wurde kein Text gelesen.'
print(kontext)

Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Millionen Bitcoins 
begrenzt. 
• Anteile an Bitcoins oder ganze Bitcoins sind immer klar einer Ad- 
resse zugeordnet.  
Informationen 
zu Transaktio- 
nen von Bit- 
coins 
• Transaktionen finden im Bitcoin-System direkt von Person zu Per- 
son statt. 
• Der zwischengeschalteter Zahlungsdienstleister (Bank)

### 3 Die Frage und die Quelle festlegen

Die Frage lautet, auf welche Gesamtmenge Bitcoin laut der Unterlage begrenzt ist.
Die Quelle besteht aus dem Dateinamen und der lesbaren Seitennummer 1.
Text, Frage und Quelle bleiben getrennte Variablen, damit sie einzeln austauschbar sind.

In [7]:
frage = 'Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?'
quelle = f'{pdf_datei.name}, S. 1'
print('Frage:', frage)
print('Quelle:', quelle)

Frage: Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 4 Die Modellschnittstelle vorbereiten

`chat_model` ist eine Funktion im Projekt. Sie erstellt den Baustein für BFH-Anfragen. Dieser Baustein heisst hier **Modellobjekt**.

Dafür verwendet die Funktion den **Adapter** `ChatOpenAI`. Ein Adapter übersetzt zwischen zwei Schnittstellen. Hier übersetzt er LangChain-Nachrichten in Anfragen an die BFH-API. Eine **API** ist die Schnittstelle, über die Programme miteinander sprechen.

Zuerst wird die Funktion nur importiert. Es wird keine Anfrage gesendet.

Dokumentation: [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai), [BFH Inference](https://infra.pages.ti.bfh.ch/mlmp/src/llm/).


In [8]:
from modul_assistent.core import chat_model


Die Verbindung braucht Modellname, API-Adresse und Schlüssel. `os.getenv` liest jeweils eine Einstellung. Der zweite Wert ist der Ersatzwert, falls die Einstellung fehlt. `.strip()` entfernt Leerzeichen am Anfang und Ende.

Für den Schlüssel gibt es keinen nutzbaren Ersatzwert. Die nächste Zelle liest die Angaben nur lokal. Sie sendet und zeigt den Schlüssel nicht an.


In [9]:
bfh_modell = os.getenv('BFH_CHAT_MODEL', 'qwen3.8-27b').strip()
bfh_adresse = os.getenv('BFH_BASE_URL', 'https://inference.proxy.ti.bfh.ch/v1').strip()
bfh_schluessel = os.getenv('BFH_API_KEY', '').strip()


### BFH Adresse und Modell kontrollieren

Die Adresse bestimmt, an welchen Dienst und welchen API-Pfad die Anfrage geht.
Der Modellname bestimmt, welches Modell dieser Dienst verwenden soll. Die beiden
Ausgaben unten zeigen diese Einstellungen, ohne den Schlüssel auszugeben.

Für den BFH-Proxy lautet die hier geprüfte API-Basisadresse
`https://inference.proxy.ti.bfh.ch/v1`. Bei diesem Proxy führt `/api/v1` zu HTTP 404.
Die Bibliotheksmeldung `OpenAIModelNotFoundError` kann auch bei einem falschen
API-Pfad erscheinen; sie beweist allein kein fehlendes Modell.


In [10]:
print('BFH-API-Adresse:', bfh_adresse)
print('BFH-Modell:', bfh_modell)

BFH-API-Adresse: https://inference.proxy.ti.bfh.ch/v1
BFH-Modell: qwen3.8-27b


Vor dem Aufruf werden leere Angaben erkannt. `if` führt einen Schritt nur aus, wenn seine Bedingung stimmt. `raise ValueError` bricht mit einer verständlichen Meldung ab.

`urlparse` zerlegt die Adresse. `scheme` ist etwa `https`; `netloc` enthält den Servernamen. Beide müssen vorhanden sein. Diese Prüfung arbeitet lokal. Sie beweist noch nicht, dass der Dienst den Schlüssel akzeptiert oder das Modell anbietet.


In [11]:
from urllib.parse import urlparse

if not bfh_schluessel:
    raise ValueError('BFH_API_KEY fehlt in der lokalen Konfiguration.')
if not bfh_modell:
    raise ValueError('BFH_CHAT_MODEL darf nicht leer sein.')
adressteile = urlparse(bfh_adresse)
if adressteile.scheme not in ('http', 'https') or not adressteile.netloc:
    raise ValueError('BFH_BASE_URL benötigt eine vollständige HTTP- oder HTTPS-Adresse.')


`chat_model()` baut nun das BFH-Modellobjekt auf. Die Klammern rufen die Funktion auf. Sie übernimmt die Einstellungen aus `core.py`.

Dort sind auch Wartezeit und begrenzte Wiederholungsversuche festgelegt. Eine geringe Zufälligkeit erleichtert Vergleiche. Sie garantiert keinen identischen Wortlaut.

Ergebnis ist `modell`, ein Baustein für spätere Anfragen. Erst `modell.invoke(...)` oder `modell.stream(...)` sendet Nachrichten.


In [12]:
modell = chat_model()


### 5 Eine erste Anfrage senden

Nun werden zwei Nachrichten vorbereitet. `SystemMessage` enthält die Regel. `HumanMessage` enthält Text und Frage. Eckige Klammern bilden eine **Liste**: mehrere Werte in einer festen Reihenfolge.

`modell.invoke(nachrichten)` sendet diese Liste an BFH und wartet auf das Ergebnis. **invoke** bedeutet hier: den Baustein mit dieser Eingabe ausführen. Das Ergebnis ist eine `AIMessage`. Ihr Feld `content` enthält den Antworttext.

**Nachrichtenliste → BFH-Aufruf → Antworttyp und Antworttext**


**Kurz überlegen:** Was wird an das BFH-Modell geschickt?

<details>
<summary>Erklärung ansehen</summary>

Die Nachrichten mit Regeln, Unterlagentext und Frage. Das Modell bekommt nicht automatisch den ganzen Projektordner.

</details>


In [13]:
from langchain_core.messages import SystemMessage, HumanMessage

nachrichten = [
    SystemMessage(content='Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.'),
    HumanMessage(content=f'Text: {kontext}\nFrage: {frage}'),
]

erste_antwort = modell.invoke(nachrichten)
print('Antworttyp:', type(erste_antwort).__name__)
print('Antwort:', erste_antwort.content)

Antworttyp: AIMessage
Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf **21 Millionen Bitcoins** begrenzt.


### 6 Was enthält eine Anschlussfrage?

Die neue Frage lautet: „Und wie entsteht der Wechselkurs?“ Ohne Verlauf steht nur dieser Satz in der Anfrage. Mit Verlauf stehen auch die Regeln, der Bitcoin-Text und die echte erste Antwort darin.

`+` verbindet hier zwei Listen. Die Schleifen zeigen jede enthaltene Nachricht einzeln. `type` benennt ihre Rolle, `content` ihren Text. `len` zählt die Nachrichten. Diese Zelle sendet keine neue Anfrage.


In [14]:
anschlussfrage = HumanMessage(content='Und wie entsteht der Wechselkurs?')
ohne_verlauf = [anschlussfrage]
mit_verlauf = nachrichten + [erste_antwort, anschlussfrage]
print('Ohne Verlauf:')
for nachricht in ohne_verlauf:
    print(nachricht.type, nachricht.content)
print('Mit Verlauf:')
for nachricht in mit_verlauf:
    print(nachricht.type, nachricht.content)
assert len(ohne_verlauf) == 1
assert len(mit_verlauf) == 4


Ohne Verlauf:
human Und wie entsteht der Wechselkurs?
Mit Verlauf:
system Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.
human Text: Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Millionen Bitcoins 
begrenzt. 
• Anteile an Bitcoins oder ganze Bitcoins sind immer klar einer Ad- 
resse zugeordnet.  
Informatio

### 7 Die Antwort schrittweise anzeigen

`invoke()` wartet auf die fertige Antwort. **Streaming** liefert schon Teile, während die Antwort entsteht. `stream()` gibt diese Teile nacheinander zurück.

Die `for`-Schleife zeigt jeden Teil sofort an. `end=''` verhindert einen neuen Absatz nach jedem Teil. `flush=True` veranlasst die sofortige Ausgabe.

Das ist eine weitere BFH-Anfrage. Sie zeigt nicht bloss die gespeicherte erste Antwort erneut an. Wortlaut und Laufzeit können sich ändern.


In [15]:
for teil in modell.stream(nachrichten):
    print(teil.content, end='', flush=True)
print()



Laut

 der Unterlage ist die Menge an Bitcoins vordef

iniert und auf **21 Millionen Bitcoins**

 begrenzt.

### 8 Tokenverbrauch und Kosten einordnen

Ein **Token** ist ein Textbaustein des Modells. Er ist nicht immer ein ganzes Wort. Auch Regeln und Kontext gehören zur Eingabe.

`usage_metadata` enthält Nutzungsdaten, wenn der Dienst sie liefert. Die Schlüssel `input_tokens` und `output_tokens` benennen Eingabe und Ausgabe. Eckige Klammern schlagen den Wert zu einem Schlüssel nach.

Die Ausgabe betrifft nur die erste vollständige Anfrage. Fehlen die Daten, wird das offen angezeigt. Dann ist kein tatsächlicher Verbrauch bekannt.


In [16]:
nutzung = erste_antwort.usage_metadata

if nutzung:
    print('Eingabe-Tokens:', nutzung['input_tokens'])
    print('Ausgabe-Tokens:', nutzung['output_tokens'])
else:
    print('Keine Tokenzahlen vom Modell gemeldet.')

Eingabe-Tokens: 579
Ausgabe-Tokens: 120


Die BFH-API meldet die verwendeten Eingabe- und Ausgabe-Tokens. Für dieses Projekt liegt kein Preis pro Token vor. Deshalb lässt sich daraus kein tatsächlicher Geldbetrag berechnen.

Allgemein hängen Kosten von der Tokenzahl und dem jeweiligen Tarif ab. Ohne bekannten Tarif bleibt diese Berechnung offen. Das bedeutet nicht automatisch, dass der Dienst kostenlos ist.


### 9 Die gewünschte Antwortform festlegen

Die bisherige Antwort ist freier Text. Für die weitere Verarbeitung soll sie zwei fest
benannte Felder enthalten: `antwort` und `quelle`. Beide Felder sollen Text enthalten.
Ein solches **Schema** beschreibt die erwartete Form der Daten.

Die kleine Klasse `Antwort` legt dieses Schema mit Pydantic fest. Ein **OutputParser** liest
anschliessend den Modelltext und prüft ihn gegen das Schema. Dadurch wird beispielsweise
erkannt, wenn ein Feld fehlt. Ob die Antwort fachlich richtig ist, bleibt eine eigene Prüfung.

In [17]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

class Antwort(BaseModel):
    antwort: str
    quelle: str

parser = PydanticOutputParser(pydantic_object=Antwort)
print('Erwartete Felder:', list(Antwort.model_fields))

Erwartete Felder: ['antwort', 'quelle']


### 10 Eine wiederverwendbare Promptvorlage erstellen

Eine **Promptvorlage** enthält Platzhalter statt fest eingebauter Inhalte.
`{kontext}`, `{quelle}` und `{frage}` werden bei einem Aufruf durch die jeweiligen Werte ersetzt.
Die Vorlage bleibt dadurch für andere Textauszüge und Fragen verwendbar.

`ChatPromptTemplate` erstellt daraus die Nachrichten. Zusätzlich beschreibt
`parser.get_format_instructions()`, wie die Antwort aufgebaut sein muss. Diese gleichbleibende
Formatvorgabe wird mit `partial()` bereits in die Vorlage eingesetzt. Beim späteren Aufruf
müssen nur noch Text, Quelle und Frage übergeben werden.

Dokumentation: [ChatPromptTemplate](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate).

In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Antworte auf Deutsch und nur anhand des Textes. '
     'Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. '
     'Ansonsten übernimm die mitgegebene Quellenangabe. {formatvorgabe}'),
    ('human', 'Text: {kontext}\nQuelle: {quelle}\nFrage: {frage}'),
]).partial(formatvorgabe='Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.')

eingaben = {'kontext': kontext, 'quelle': quelle, 'frage': frage}

Vor einer weiteren Modellanfrage lohnt sich ein Blick auf die ausgefüllte Vorlage.
`prompt.invoke(eingaben)` setzt die Werte ein; `to_string()` macht die entstandenen Nachrichten
lesbar. Dieser Schritt ruft noch kein Modell auf. Fehlende oder falsch zugeordnete Werte lassen
sich dadurch früh erkennen.

In [19]:
vorbereiteter_prompt = prompt.invoke(eingaben)
print(vorbereiteter_prompt.to_string())

System: Antworte auf Deutsch und nur anhand des Textes. Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. Ansonsten übernimm die mitgegebene Quellenangabe. Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.
Human: Text: Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Mill

### 11 Prompt Modell und Parser verbinden

Eine **Chain** ist eine Kette von Arbeitsschritten. Hier sind es drei:

**Eingabe-Dictionary → Promptvorlage → Modell → Parser → Ergebnis mit zwei Feldern**

LangChain nennt diese Schreibweise **LCEL**: LangChain Expression Language. Der Strich `|` gibt das Ergebnis links an den nächsten Baustein rechts weiter.

Das Modell soll **JSON** ausgeben. JSON ist ein Textformat für benannte Daten, etwa `{"antwort": "Text", "quelle": "Datei"}`. Der Parser liest diesen Text und prüft die Felder. `chain.invoke(eingaben)` führt die ganze Kette aus und sendet dabei eine neue BFH-Anfrage.


`bind` ergänzt eine feste Einstellung für die folgenden Aufrufe. Hier bittet sie die BFH-Schnittstelle um ein JSON-Objekt. Der Parser prüft danach die beiden Felder.


**Kurz überlegen:** Welcher Schritt erzeugt die Antwort?

<details>
<summary>Erklärung ansehen</summary>

Das Modell. Die Vorlage stellt die Nachrichten zusammen. Der Parser liest danach die Antwort und prüft die Felder.

</details>


In [20]:
struktur_modell = modell.bind(response_format={'type': 'json_object'})
chain = prompt | struktur_modell | parser
ergebnis = chain.invoke(eingaben)

print('Antwort:', ergebnis.antwort)
print('Quelle:', ergebnis.quelle)
assert isinstance(ergebnis, Antwort)


Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 12 Den Datenfluss Schritt für Schritt untersuchen

Die kurze Schreibweise mit `|` fasst mehrere Schritte zusammen. Zur Fehlersuche lassen sie
sich einzeln ausführen. Die folgende Zelle zeigt die drei Zwischenergebnisse.

Für einen Debugger-Durchlauf kann ein Haltepunkt bei `nachrichtenwert` gesetzt werden.
Beim schrittweisen Ausführen lässt sich verfolgen, wie aus dem Dictionary `eingaben` zuerst
Nachrichten, danach Modelltext und schliesslich eine geprüfte `Antwort` werden. So ist bei einem
Fehler erkennbar, in welchem Schritt er entsteht.

In [21]:
nachrichtenwert = prompt.invoke(eingaben)
modellantwort = struktur_modell.invoke(nachrichtenwert)
gepruefte_antwort = parser.invoke(modellantwort)

print('Nach der Vorlage:', type(nachrichtenwert).__name__)
print('Nach dem Modell:', modellantwort.content)
print('Nach dem Parser:', gepruefte_antwort.model_dump())

Nach der Vorlage: ChatPromptValue
Nach dem Modell: {
  "antwort": "Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.",
  "quelle": "Eigenschaften-Bitcoin.pdf, S. 1"
}
Nach dem Parser: {'antwort': 'Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.', 'quelle': 'Eigenschaften-Bitcoin.pdf, S. 1'}


## Häufige Fehler

### Eine benötigte Promptvariable fehlt

Die Vorlage benötigt `kontext`, `quelle` und `frage`. Die nächste Zelle übergibt absichtlich
nur die Frage. Dadurch fehlen zwei Werte. Bereits das Ausfüllen der Vorlage bricht mit einem
`KeyError` ab; es wird keine Modellanfrage gesendet.

`try` und `except` fangen hier ausschliesslich den erwarteten Fehlertyp ab, damit der geplante
Fehler sichtbar wird und das Notebook trotzdem weiterläuft. Die Fehlermeldung nennt die
fehlenden Variablen.

In [22]:
try:
    prompt.invoke({'frage': frage})
except KeyError as fehler:
    print('Erwarteter Fehler:', type(fehler).__name__)
    print(str(fehler)[:450])
else:
    raise AssertionError('Der erwartete Fehler ist nicht aufgetreten.')

Erwarteter Fehler: KeyError
"Input to ChatPromptTemplate is missing variables {'quelle', 'kontext'}.  Expected: ['frage', 'kontext', 'quelle'] Received: ['frage']\nNote: if you intended {quelle} to be part of the string and not a variable, please escape it with double curly braces like: '{{quelle}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


### Die Eingaben vervollständigen

Die Korrektur verwendet das vollständige Dictionary `eingaben`. Es enthält alle drei benötigten Schlüssel.

Die Vorlage wird erneut befüllt. Die Prüfung erwartet zwei Nachrichten: System und Human. Eine zusätzliche Modellanfrage ist dafür nicht nötig.


In [23]:
korrigierter_prompt = prompt.invoke(eingaben)
print('Die Vorlage wurde vollständig ausgefüllt.')
assert len(korrigierter_prompt.to_messages()) == 2

Die Vorlage wurde vollständig ausgefüllt.


## Prompt Logbuch

### Warum steht dieser Abschnitt hier?

Die Aufgabenstellung verlangt, die Entwicklung eines Prompts zu dokumentieren: ein zuerst selbst geschriebener Auftrag, die später mit KI überarbeitete Fassung und eine kurze Erklärung der Änderung. So wird sichtbar, was einen genaueren Prompt ausmacht.

Ein **Prompt** besteht aus Anweisungen, Frage und mitgegebenen Informationen. Hier werden die erste dokumentierte Fassung und die aktuelle Endfassung gegenübergestellt. „Endfassung“ bedeutet die derzeit verwendete Version, nicht eine perfekte Anweisung.

n.

### 1 Erste dokumentierte Fassung

Die erste Fassung steht bei der ersten Modellanfrage. Die Endfassung steht in der späteren Promptvorlage. Beide sind im Notebook als Code vorhanden und wurden technisch über BFH ausgeführt.

```text

Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.

```

### 2 Aktuelle Endfassung

```text

Antworte auf Deutsch und nur anhand des Textes. Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. Ansonsten übernimm die mitgegebene Quellenangabe. Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.

```

`{kontext}` ist der gelesene PDF-Text, `{quelle}` enthält Datei und Seite, `{frage}` die aktuelle Frage. Die Beispiel-Frage betrifft die Begrenzung von Bitcoin.

### 3 Was wurde genauer?

Die erste Fassung begrenzt die Antwort auf den Text. Die Endfassung legt zusätzlich Quelle, Verhalten bei fehlender Information und die zwei JSON-Felder fest. So weiss das Modell genauer, was erwartet wird. Der Parser prüft die Felder; die fachliche Aussage muss trotzdem am Text geprüft werden.

Die Beschreibung erklärt die Änderungen an der Anweisung. Sie behauptet keine gemessene Verbesserung zwischen zwei Fassungen, wenn ein solcher Vergleich nicht ausgeführt wurde.


## Zusammenfassung

Aus einer einzelnen Frage ist eine kleine Verarbeitungskette entstanden. Der Text und die Frage
werden als Nachrichten übergeben. Eine Promptvorlage hält die wiederverwendbaren Anweisungen fest.
Modell und Parser liefern daraus eine Antwort mit Quellenangabe.

Die bisherigen Schritte beantworten eine einzelne Anfrage. Ein Gesprächsverlauf wird noch nicht
automatisch gespeichert. Notebook 02 ergänzt diese Fähigkeit und vergleicht Möglichkeiten,
frühere Nachrichten bei einer Anschlussfrage erneut zu berücksichtigen.

### Lernziele überprüfen

- Welche Angaben stehen in der Systemnachricht und welche in der Human-Nachricht? Was fehlt einer Anschlussfrage ohne Verlauf?
- Welche drei Schritte verbindet die Chain, und welche Art von Ergebnis gibt jeder Schritt weiter?
- Was unterscheidet `invoke()` von `stream()`?
- Welche Tokenzahlen wurden tatsächlich gemeldet? Welche Werte der Kostenrechnung sind lediglich Annahmen?

## Übungsaufgabe

Formuliere zwei neue Fragen zur Bitcoin-Seite. Eine soll der Text beantworten können. Für die andere soll eine Angabe fehlen. Ein Beispiel für fehlende Information ist der heutige Bitcoin-Kurs.

Schreibe für einen neuen Fall zuerst selbst einen Prompt. Lass ihn erst danach mit KI-Hilfe überarbeiten. Führe die tatsächlichen Fassungen aus und ergänze das Prompt-Logbuch.

Prüfe am PDF-Text: Stimmt die Antwort? Stimmt die Quelle? Wird fehlende Information benannt? Die letzte Codezelle bleibt für die eigene Lösung leer.
